## Jamie Data Cleaning

In [1]:
import json
import re
import html
import pandas as pd
from pathlib import Path

## 1. Project vision and question

Project goal: An Amazon Appliances category manager wants to identify product listings that may need clearer information.

Business question: What product details do customers bring up repeatedly in reviews, and are those details clear in the listing?

Main metric: For each product, the share of reviews that mention a particular detail, such as size or fit.

In [2]:
from pathlib import Path
import pandas as pd

data_path = Path("dataset/Appliances_sample.jsonl")
if not data_path.exists():
    data_path = Path("Appliances_sample.jsonl")

data = pd.read_json(data_path, lines=True)
data.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Work great,work great. use a new one every month,[],B01N0TQ0OH,B01N0TQ0OH,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2018-02-22 16:31:48.692,0,True
1,5,excellent product,Little on the thin side,[],B07DD2DMXB,B07DD37QPZ,AHWWLSPCJMALVHDDVSUGICL6RUCA,2022-10-02 21:41:03.446,0,True
2,5,Happy customer!,"Quick delivery, fixed the issue!",[],B082W3Z9YK,B082W3Z9YK,AHZIJGKEWRTAEOZ673G5B3SNXEGQ,2020-12-06 03:30:35.363,0,True
3,5,Amazing value,I wasn't sure whether these were worth it or n...,[],B078W2BJY8,B078W2BJY8,AFGUPTDFAWOHHL4LZDV27ERDNOYQ,2018-08-12 20:03:04.306,0,True
4,5,Dryer parts,Easy to install got the product expected to re...,[],B08C9LPCQV,B08C9LPCQV,AELFJFAXQERUSMTXJQ6SYFFRDWMA,2021-05-05 01:03:23.754,0,True


In [3]:
print(f"Rows: {data.shape[0]:,}")
print(f"Columns: {data.shape[1]:,}")
print("Column names:", data.columns.tolist())
data.info()

Rows: 1,000
Columns: 10
Column names: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   rating             1000 non-null   int64         
 1   title              1000 non-null   object        
 2   text               1000 non-null   object        
 3   images             1000 non-null   object        
 4   asin               1000 non-null   object        
 5   parent_asin        1000 non-null   object        
 6   user_id            1000 non-null   object        
 7   timestamp          1000 non-null   datetime64[ns]
 8   helpful_vote       1000 non-null   int64         
 9   verified_purchase  1000 non-null   bool          
dtypes: bool(1), datetime64[ns](1), int64(2), object(6)
memory usage: 71.4

## 2. Define one row

Each row is one customer's review of a product. A customer can review more than one product, and a product can have reviews from many customers, so neither `user_id` nor `asin` identifies a review by itself.

There isn't a review ID in this file, so I added `original_row_number` to keep track of where each review came from.

In [4]:
data["original_row_number"] = range(1, len(data) + 1)

print("Rows:", len(data))
print("Unique original row numbers:", data["original_row_number"].nunique())

Rows: 1000
Unique original row numbers: 1000


## 3. Preserve the source

I used 1,000 reviews from the Appliances category of McAuley Lab's Amazon Reviews 2023. My team put the sample in `dataset/Appliances_sample.jsonl` because the full file was too large for our repository, and our professor approved using the sample.

I load the file with `pd.read_json(..., lines=True)` and save my cleaned data separately, so the original sample stays unchanged.

## 4. Inspect columns and types

Failed timestamp conversions: 0 of 1,000. 

The original timestamp is an integer in milliseconds. I converted it to a UTC date and time which makes the review dates usable. Ratings and helpful votes are numeric, verified purchase is Boolean, and product and user IDs remain text.

In [5]:
raw_types = pd.read_json(data_path, lines=True, convert_dates=False)

print(raw_types.dtypes)

review_dates = pd.to_datetime(
    raw_types["timestamp"],
    unit="ms",
    utc=True,
    errors="coerce"
)

print("Failed timestamp conversions:", review_dates.isna().sum())
print("Earliest review:", review_dates.min())
print("Latest review:", review_dates.max())

rating                int64
title                object
text                 object
images               object
asin                 object
parent_asin          object
user_id              object
timestamp             int64
helpful_vote          int64
verified_purchase      bool
dtype: object
Failed timestamp conversions: 0
Earliest review: 2009-09-27 16:08:41+00:00
Latest review: 2023-03-14 02:44:35.407000+00:00


## 5. Standardize text

I removed formatting like "<br />" and extra spaces from review titles and text so they’re easier to read. I left the customers’ wording as it was and didn’t change any product or user IDs.

Result: 13 titles and 314 review texts changed.

Before (original row 4):
"...original branded filters.<br /><br />I can happily report..."

After:
"...original branded filters. I can happily report..."

This removes formatting markup without changing the content of the review.

In [6]:
cleaned_data = data.copy()

def clean_review_text(value):
    value = re.sub(r"<br\s*/?>", " ", value, flags=re.IGNORECASE)
    value = html.unescape(value)
    return " ".join(value.split())

for column in ["title", "text"]:
    cleaned_data[column] = cleaned_data[column].map(clean_review_text)
    changed = (data[column] != cleaned_data[column]).sum()
    print(f"{column}: {changed} rows changed")

print("\nBefore:", data.loc[3, "text"])
print("\nAfter: ", cleaned_data.loc[3, "text"])

title: 13 rows changed
text: 314 rows changed

Before: I wasn't sure whether these were worth it or not, given the cost compared to the original branded filters.<br /><br />I can happily report that these are a great value and work every bit as good as the original. If you are on the fence worrying whether these are worth it- I can assure you they are.

After:  I wasn't sure whether these were worth it or not, given the cost compared to the original branded filters. I can happily report that these are a great value and work every bit as good as the original. If you are on the fence worrying whether these are worth it- I can assure you they are.


## 6. Investigate duplicates

I found 0 exact duplicate reviews, so I removed 0 rows.
Some user IDs and product IDs appear more than once because users can review multiple products and products can receive multiple reviews. I kept those reviews.

In [7]:
review_keys = raw_types.apply(
    lambda row: json.dumps(row.to_dict(), sort_keys=True),
    axis=1
)

print("Exact duplicate reviews:", review_keys.duplicated().sum())
print("Repeated user IDs:", raw_types["user_id"].duplicated().sum())
print("Repeated product IDs:", raw_types["asin"].duplicated().sum())

print(
    "Repeated user + product pairs:",
    raw_types.duplicated(subset=["user_id", "asin"]).sum()
)

Exact duplicate reviews: 0
Repeated user IDs: 375
Repeated product IDs: 130
Repeated user + product pairs: 0


## 7. Handle missing data

All 10 columns have 0 missing values before and after cleaning. There are also 0 blank titles and 0 blank review texts. I did not fill or drop any values. An empty images list (`[]`) means the customer did not attach an image; I keep it as a valid value.

In [8]:
columns = raw_types.columns

missing_before = raw_types[columns].isna().sum()
missing_after = cleaned_data[columns].isna().sum()

print(
    pd.DataFrame({
        "before": missing_before,
        "after": missing_after
    })
)

print("Blank titles:", cleaned_data["title"].str.strip().eq("").sum())
print("Blank review texts:", cleaned_data["text"].str.strip().eq("").sum())

                   before  after
rating                  0      0
title                   0      0
text                    0      0
images                  0      0
asin                    0      0
parent_asin             0      0
user_id                 0      0
timestamp               0      0
helpful_vote            0      0
verified_purchase       0      0
Blank titles: 0
Blank review texts: 0


## 8. Check suspicious values

There are 0 ratings outside 1–5,  0 helpful vote counts that were negative, and 0 dates that failed conversion.

In row 6, the original sample has a 5-star rating but the title says "DO NOT purchase." I flagged this inconsistency for manual review. I kept the original rating and text because there is no evidence showing which review the customer intended to make.

In [9]:
print("Invalid ratings:", (~raw_types["rating"].isin([1, 2, 3, 4, 5])).sum())
print("Negative helpful votes:", (raw_types["helpful_vote"] < 0).sum())
print("Dates that failed conversion:", review_dates.isna().sum())

#manual check for unusual case
print("\nReview to inspect:")
print(raw_types.loc[5, ["rating", "title", "text", "asin"]])

Invalid ratings: 0
Negative helpful votes: 0
Dates that failed conversion: 0

Review to inspect:
rating                                                    5
title                     DO NOT purchase this ice machine.
text      After buying this ice machine just 15 months a...
asin                                             B08D6RFV6D
Name: 5, dtype: object


## 9. Verify merges

Not applicable. I only cleaned the Appliances review file for this assignment. I did not combine it with another dataset, so there are no unmatched records or changes in row count from a merge to check.

## 10. Reconcile changes

I started with 1,000 reviews and ended with 1,000, so no rows were removed. I found no exact duplicates or missing values that required removing a review. The changes were to text formatting and the way timestamps are displayed. Rating and total helpful vote counts stayed the same.

In [10]:
print("Rows before:", len(raw_types))
print("Rows after:", len(cleaned_data))
print("Rows removed:", len(raw_types) - len(cleaned_data))

print("Rating total before:", raw_types["rating"].sum())
print("Rating total after:", cleaned_data["rating"].sum())

print("Helpful votes before:", raw_types["helpful_vote"].sum())
print("Helpful votes after:", cleaned_data["helpful_vote"].sum())

Rows before: 1000
Rows after: 1000
Rows removed: 0
Rating total before: 4406
Rating total after: 4406
Helpful votes before: 1229
Helpful votes after: 1229


## 11. Check calculations

The average rating uses all 1,000 reviews because none has a missing rating. The average is 4.406 stars. If a rating were missing, I would leave that review out of the rating average rather than treat it as 0. No parent_asin values are missing, so every review has a product group.

In [11]:
print("Reviews in rating average:", cleaned_data["rating"].notna().sum())
print("Average rating:", cleaned_data["rating"].mean())
print("Missing parent product IDs:", cleaned_data["parent_asin"].isna().sum())

Reviews in rating average: 1000
Average rating: 4.406
Missing parent product IDs: 0


## 12. Verify five reviews

I selected five reviews that cover an unchanged record, an ID I needed to preserve, HTML formatting, a rating/text inconsistency, and an image attachment.

| Original row | What I checked | Expected result and reason |
|---|---|---|
| 1 | Title | "Work great" stays the same; it has no formatting to remove. |
| 2 | `parent_asin` | `B07DD37QPZ` stays the same; IDs should not be edited. |
| 4 | Review text | The `<br /><br />` after "filters." becomes a space. |
| 6 | Rating | 5 stays 5, despite the negative title; I cannot verify which is wrong. |
| 98 | Images | The review still has one image; text cleaning should not affect it. |

In [15]:
checks = [
    (1, "title", raw_types.loc[0, "title"],
     "Work great", cleaned_data.loc[0, "title"]),

    (2, "parent_asin", raw_types.loc[1, "parent_asin"],
     "B07DD37QPZ", cleaned_data.loc[1, "parent_asin"]),

    (4, "text near HTML break",
     "filters.<br /><br />I can",
     "filters. I can",
     "filters. I can" if "filters. I can" in cleaned_data.loc[3, "text"]
     else "Expected phrase not found"),

    (6, "rating", raw_types.loc[5, "rating"],
     5, cleaned_data.loc[5, "rating"]),

    (98, "number of images", len(raw_types.loc[97, "images"]),
     1, len(cleaned_data.loc[97, "images"])),
]

for row, field, original, expected, actual in checks:
    print(f"Row {row} — {field}")
    print(f"  Original: {original}")
    print(f"  Expected: {expected}")
    print(f"  Actual:   {actual}")
    print(f"  Match:    {expected == actual}\n")

Row 1 — title
  Original: Work great
  Expected: Work great
  Actual:   Work great
  Match:    True

Row 2 — parent_asin
  Original: B07DD37QPZ
  Expected: B07DD37QPZ
  Actual:   B07DD37QPZ
  Match:    True

Row 4 — text near HTML break
  Original: filters.<br /><br />I can
  Expected: filters. I can
  Actual:   filters. I can
  Match:    True

Row 6 — rating
  Original: 5
  Expected: 5
  Actual:   5
  Match:    True

Row 98 — number of images
  Original: 1
  Expected: 1
  Actual:   1
  Match:    True



## 13. Test reproducibility

I restarted the kernel and ran all cells from the top. The notebook
loaded `dataset/Appliances_sample.jsonl` and created `output/Appliances_cleaned.jsonl` with 1,000 reviews. All cells ran
without errors.

In [13]:
from pathlib import Path

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

cleaned_data.to_json(
    output_dir / "Appliances_cleaned.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
    date_format="iso"
)

print("Saved:", output_dir / "Appliances_cleaned.jsonl")
print("Rows saved:", len(cleaned_data))

Saved: output/Appliances_cleaned.jsonl
Rows saved: 1000


## 14. Remaining limitations

I only have reviews, not the product listings. A review may mention a
missing detail, but I can't confirm a listing mismatch until I compare
it with the listing. I would also need to check that each review matches
the right product.

One review has a 5-star rating and a negative title. I can't verify what the actual customer's intended rating. If the 5-star rating was a mistake, it could make the product’s average rating too high. I left it unchanged.

I included the five reviews I checked by hand and selected 45 more with a fixed random seed (42), so rerunning the notebook produces the same example file.

In [ ]:
checked_rows = [0, 1, 3, 5, 97]  # The five reviews from item 12
other_rows = cleaned_data.drop(index=checked_rows).sample(
    n=45, random_state=42
)

example_rows = pd.concat([
    cleaned_data.loc[checked_rows],
    other_rows
]).sort_index()

example_rows.to_json(
    output_dir / "Appliances_cleaned_50_examples.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
    date_format="iso"
)

print("Example rows saved:", len(example_rows))